# Fine-Tune AI Fitness Coach using QLoRA

This notebook demonstrates how to fine-tune `google/gemma-2b-it` using the fitness dataset we prepared (`dataset.jsonl`).

**Note:** Please run this on Google Colab or Kaggle Notebooks with a T4 or P100 GPU enabled.

In [ ]:
!pip install -q -U transformers datasets peft trl bitsandbytes accelerate

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

In [ ]:
# 1. Load Dataset
# Make sure to upload your dataset.jsonl to the Colab/Kaggle environment
dataset = load_dataset('json', data_files='dataset.jsonl', split='train')

def format_instruction(example):
    return f"<start_of_turn>user\n{example['instruction']}<end_of_turn>\n<start_of_turn>model\n{example['response']}<end_of_turn>"

# You would use a formatting function inside the SFTTrainer below.

In [ ]:
# 2. Load Model and Tokenizer
model_id = "google/gemma-2b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")
model = prepare_model_for_kbit_training(model)

In [ ]:
# 3. Configure LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "o_proj", "k_proj", "v_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

In [ ]:
# 4. Training
sft_config = SFTConfig(
    output_dir="./fitness-coach-model",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    save_steps=50,
    logging_steps=10,
    learning_rate=2e-4,
    max_grad_norm=0.3,
    max_steps=200, # Increase for real training
    warmup_ratio=0.03,
    lr_scheduler_type="constant",
    dataset_text_field="text"
)

# Format dataset text for SFT
dataset = dataset.map(lambda x: {"text": format_instruction(x)})

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=sft_config,
)

trainer.train()

In [ ]:
# 5. Save the tuned adapter
trainer.model.save_pretrained("fitness-coach-lora-adapter")
tokenizer.save_pretrained("fitness-coach-lora-adapter")
print("Training complete! Adapter saved.")